In [ ]:
global_noise_std = 0.075
train_mode = 'recurrent'      # all, input, recurrent
num_trials = 100
# let set some parameters
n_eval_every = 2
n_save_train_snapshot_every = 1

In [ ]:
# traing the RNN to learn a first decoder axis, then remapping to a second decoder axis that is orthogonal to the first,
#  and retraining. This is the main simulation for the paper.
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from core import (
    RNNConfig,
    MotorCortexRNN,
    BCIDecoderConfig,
    PopulationBCIDecoder,
    TrialInputConfig,
    CursorTargetConfig,
    TrainerConfig,
    BCITrainer,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


# Build model
rnn = MotorCortexRNN(RNNConfig(n_inp=10, n_rec=200, noise_std=global_noise_std, device="cpu"))
decoder = PopulationBCIDecoder(BCIDecoderConfig(n_rec=200, n_bci=50, device="cpu"))

trial_cfg = TrialInputConfig(
    n_inp=10,
    t_baseline=20,
    t_task=20,
    t_late=20,
    baseline_scale=0.3,
    task_scale=1.0,
    late_scale=0.6,
    input_noise_std=global_noise_std,
    device="cpu",
)


target_cfg = CursorTargetConfig(
    baseline_target=0.0,
    late_target=1.0,
    task_target_mode="linear",
    baseline_weight=0.5,
    task_weight=1.0,
    late_weight=0.75,
    device="cpu",
)

trainer_cfg = TrainerConfig(
    train_mode= train_mode,      # all, input, recurrent
    lr=1e-3,
    grad_clip_norm=1.0,
    device="cpu",
    freeze_trial_inputs=True,
    input_scaling_mode="cursor_mean",
    input_scale_gain=0.2,
)

trainer = BCITrainer(rnn, decoder, trial_cfg, target_cfg, trainer_cfg)

# Save first decoder axis
axis1 = trainer.decoder.axis.detach().clone()


# Phase 1: learn decoder 1
trainer.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_1",
    start_step=0,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=30,
    save_train_snapshot_every=n_save_train_snapshot_every,
    save_eval_snapshots=True,
)

'''# Make a second decoder axis on the SAME 50 neurons
axis2 = torch.zeros_like(axis1)
idx = trainer.decoder.neuron_indices
axis2[idx] = torch.randn(len(idx))
axis2 = axis2 / axis2.norm()'''

# Make a second decoder axis on the SAME 50 neurons,
# and force it to be orthogonal to axis1
axis2 = torch.zeros_like(axis1)
idx = trainer.decoder.neuron_indices

v = torch.randn(len(idx), device=axis1.device)
axis2[idx] = v

# subtract projection onto axis1
axis2 = axis2 - torch.dot(axis2, axis1) * axis1

# normalize
axis2 = axis2 / axis2.norm()

print("dot(axis1, axis2) =", torch.dot(axis1, axis2).item())



# Remap: same model, same learned weights, new decoder
trainer.set_decoder_axis(axis2)

# Phase 2: relearn new decoder
trainer.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_2_remap",
    start_step=num_trials,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=10,
    save_train_snapshot_every=n_save_train_snapshot_every,   # save every remap trial if you want signatures
    save_eval_snapshots=True,
)

In [ ]:
# sanity check plots
import numpy as np
import matplotlib.pyplot as plt

hist = trainer.history.to_dict()
snaps = trainer.history.snapshots
fig_size = (6, 2.5)

# ----------------------------
# 1) Loss curve across phases
# ----------------------------
train_steps = [s for s, m in zip(hist["step"], hist["mode"]) if m == "train"]
train_loss = [l for l, m in zip(hist["loss"], hist["mode"]) if m == "train"]

eval_steps = [s for s, m in zip(hist["step"], hist["mode"]) if m == "eval"]
eval_loss = [l for l, m in zip(hist["loss"], hist["mode"]) if m == "eval"]

plt.figure(figsize=fig_size)
plt.plot(train_steps, train_loss, label="train")
plt.plot(eval_steps, eval_loss, label="eval")
plt.axvline(num_trials, linestyle="--", label="remap")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training loss across decoder switch")
plt.legend()
plt.tight_layout()
plt.show()


# ----------------------------
# 2) Cursor vs target:
#    before remap / early remap / late remap
# ----------------------------
def get_snapshot(phase=None, step=None, which="last"):
    candidates = snaps
    if phase is not None:
        candidates = [s for s in candidates if s["phase"] == phase]
    if step is not None:
        candidates = [s for s in candidates if s["step"] == step]

    if len(candidates) == 0:
        raise ValueError("No matching n snapshot found.")

    if which == "first":
        return sorted(candidates, key=lambda s: s["step"])[0]
    elif which == "last":
        return sorted(candidates, key=lambda s: s["step"])[-1]
    else:
        raise ValueError("which must be 'first' or 'last'")


snap_before = get_snapshot(phase="decoder_1", which="last")
snap_early = get_snapshot(phase="decoder_2_remap", which="first")
snap_late = get_snapshot(phase="decoder_2_remap", which="last")

fig, axes = plt.subplots(1, 3, figsize=fig_size, sharey=True)

for ax, snap, title in zip(
    axes,
    [snap_before, snap_early, snap_late],
    ["Before remap", "Early remap", "Late remap"],
):
    cursor = snap["cursor"].numpy()
    target = snap["target"].numpy()

    ax.plot(cursor, label="cursor")
    ax.plot(target, label="target")
    ax.set_title(f"{title} (step {snap['step']})", fontsize=10)
    ax.set_xlabel("Time")

axes[0].set_ylabel("Value")
axes[0].legend()
plt.tight_layout()
plt.show()


# ----------------------------
# 3) Epoch-mean cursor over time
# ----------------------------
steps_train = np.array([r["step"] for r in trainer.history.records if r["mode"] == "train"])
base_train = np.array([r["baseline_cursor_mean"] for r in trainer.history.records if r["mode"] == "train"])
task_train = np.array([r["task_cursor_mean"] for r in trainer.history.records if r["mode"] == "train"])
late_train = np.array([r["late_cursor_mean"] for r in trainer.history.records if r["mode"] == "train"])

plt.figure(figsize=fig_size)
plt.plot(steps_train, base_train, label="baseline")
plt.plot(steps_train, task_train, label="task")
plt.plot(steps_train, late_train, label="late")
plt.axvline(num_trials, linestyle="--", label="remap")
plt.xlabel("Step")
plt.ylabel("Mean cursor")
plt.title("Epoch-wise mean cursor across training")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# User settings
# =========================
axis_shift_step = num_trials
use_bci_only = True          # True: only 50 BCI neurons, False: all neurons
window_size_trials = 5
stride_trials = window_size_trials
pca_dim_m = 10
global_pca_components = 3
every_k_trials_global = 10
every_k_trials_phase = 5


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from core import (
    analyze_geometry_windows,
    fit_global_pca,
    fit_phase_pca,
    fit_phase_endpoint_cloud_pca,
)


# =========================
# Run analyses
# =========================
analysis_all = analyze_geometry_windows(
    snapshots=trainer.history.snapshots,
    phase=None,
    use_bci_only=use_bci_only,
    window_size_trials=window_size_trials,
    stride_trials=stride_trials,
    pca_dim_m=pca_dim_m,
)


# =========================
# 1) Plot ALL numeric metrics
# =========================
wm = analysis_all["window_metrics"]
ca = analysis_all["consecutive_alignment"]

x_win = np.array([m["start_step"] for m in wm])
d95 = np.array([m["effective_dim_95"] for m in wm])
var_amb = np.array([m["trajectory_variance_ambient"] for m in wm])
var_pca = np.array([m["trajectory_variance_pca"] for m in wm])
align_cos = np.array([m["decoder_alignment_pointwise_cosine"] for m in wm], dtype=float)
align_mean_ang = np.array([m["decoder_alignment_mean_angle_deg"] for m in wm], dtype=float)

fig, axes = plt.subplots(3, 2, figsize=(15, 14))

# Effective dimensionality
axes[0, 0].plot(x_win, d95, marker="o")
axes[0, 0].axvline(axis_shift_step, linestyle="--")
axes[0, 0].set_title("Effective dimensionality (95% variance)")
axes[0, 0].set_xlabel("Window start step")
axes[0, 0].set_ylabel("Dimensionality")

# Ambient and PCA trajectory variance
axes[0, 1].plot(x_win, var_amb, marker="o", label="Ambient variance")
axes[0, 1].axvline(axis_shift_step, linestyle="--")
axes[0, 1].set_title("Trajectory variance")
axes[0, 1].set_xlabel("Window start step")
axes[0, 1].set_ylabel("Variance")
axes[0, 1].legend()

# Decoder alignment cosine
axes[1, 0].plot(x_win, align_cos, marker="o", label="Cosine")
axes[1, 0].axvline(axis_shift_step, linestyle="--")
axes[1, 0].set_title("Decoder-axis cosine alignment with trajectory")
axes[1, 0].set_xlabel("Window start step")
axes[1, 0].set_ylabel("Cosine alignment")
axes[1, 0].legend()

# Decoder alignment mean angle
axes[1, 1].plot(x_win, align_mean_ang, marker="o", label="Mean angle (deg)")
axes[1, 1].axvline(axis_shift_step, linestyle="--")
axes[1, 1].set_title("Decoder-axis mean angle alignment with trajectory")
axes[1, 1].set_xlabel("Window start step")
axes[1, 1].set_ylabel("Mean angle (deg)")
axes[1, 1].legend()

# Principal angles between consecutive manifolds
if len(ca) > 0:
    x_ca = np.array([a["end_step_2"] for a in ca])
    min_ang = np.array([a["min_angle_deg"] for a in ca])
    max_ang = np.array([a["max_angle_deg"] for a in ca])
    mean_ang = np.array([a["mean_angle_deg"] for a in ca])

    axes[2, 0].plot(x_ca, min_ang, marker="o", label="Min angle")
    axes[2, 0].axvline(axis_shift_step, linestyle="--")
    axes[2, 0].set_title("Consecutive manifold principal angles")
    axes[2, 0].set_xlabel("Next window start step")
    axes[2, 0].set_ylabel("Angle (deg)")
    axes[2, 0].legend()
else:
    axes[2, 0].text(0.5, 0.5, "Not enough windows", ha="center", va="center")
    axes[2, 0].set_axis_off()

# Z-scored summary overlay
def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    s = np.nanstd(x)
    return (x - np.nanmean(x)) / s if s > 0 else np.zeros_like(x)

axes[2, 1].plot(x_win, zscore_safe(d95), marker="o", label="d95")
axes[2, 1].plot(x_win, zscore_safe(var_amb), marker="o", label="var ambient")
axes[2, 1].plot(x_win, zscore_safe(var_pca), marker="o", label="var pca")
axes[2, 1].plot(x_win, zscore_safe(align_cos), marker="o", label="align cosine")
if len(ca) > 0:
    axes[2, 1].plot(x_ca, zscore_safe(max_ang), marker="o", label="max princ angle")
axes[2, 1].axvline(axis_shift_step, linestyle="--")
axes[2, 1].set_title("All metrics (z-scored)")
axes[2, 1].set_xlabel("Step")
axes[2, 1].legend()

plt.tight_layout()
plt.show()





In [ ]:
global_pca = fit_global_pca(
    snapshots=trainer.history.snapshots,
    phase=None,
    use_bci_only=use_bci_only,
    n_components=global_pca_components,
)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from core import (
    analyze_geometry_windows,
    fit_global_pca,
)

# --------------------------------------------------
# User-set parameters
# --------------------------------------------------
axis_shift_step = 100          # mark decoder switch here
window_size_trials = 10
stride_trials = 1
last_k_time = 20
pca_dim_m = 10
use_bci_only = True            # True -> 50 BCI neurons, False -> all neurons
every_k_trials = 10            # for global PCA trajectory plotting


# --------------------------------------------------
# 1) Windowed geometry metrics across ALL snapshots
# --------------------------------------------------
analysis_all = analyze_geometry_windows(
    snapshots=trainer.history.snapshots,
    phase=None,                        # use all saved phases
    use_bci_only=use_bci_only,
    window_size_trials=window_size_trials,
    stride_trials=stride_trials,
    last_k_time=last_k_time,
    pca_dim_m=pca_dim_m,
)

window_metrics = analysis_all["window_metrics"]
align_metrics = analysis_all["consecutive_alignment"]

x_win = np.array([m["start_step"] for m in window_metrics])
d95 = np.array([m["effective_dim_95"] for m in window_metrics])
var_amb = np.array([m["trajectory_variance_ambient"] for m in window_metrics])
var_pca = np.array([m["trajectory_variance_pca"] for m in window_metrics])

if len(align_metrics) > 0:
    x_ang = np.array([a["start_step_2"] for a in align_metrics])
    min_ang = np.array([a["min_angle_deg"] for a in align_metrics])
    max_ang = np.array([a["max_angle_deg"] for a in align_metrics])
    mean_ang = np.array([a["mean_angle_deg"] for a in align_metrics])
else:
    x_ang = np.array([])
    min_ang = np.array([])
    max_ang = np.array([])
    mean_ang = np.array([])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Effective dimensionality
axes[0, 0].plot(x_win, d95, marker="o")
axes[0, 0].axvline(axis_shift_step, linestyle="--")
axes[0, 0].set_title("Effective dimensionality (95% variance)")
axes[0, 0].set_xlabel("Window start step")
axes[0, 0].set_ylabel("Dimensionality")

# Trajectory variance
axes[0, 1].plot(x_win, var_amb, marker="o", label="Ambient")
axes[0, 1].plot(x_win, var_pca, marker="o", label=f"PCA (m={pca_dim_m})")
axes[0, 1].axvline(axis_shift_step, linestyle="--")
axes[0, 1].set_title("Trajectory variance over trial windows")
axes[0, 1].set_xlabel("Window start step")
axes[0, 1].set_ylabel("Variance")
axes[0, 1].legend()

# Principal angles
if len(x_ang) > 0:
    axes[1, 0].plot(x_ang, min_ang, marker="o", label="Min angle")
    #axes[1, 0].plot(x_ang, max_ang, marker="o", label="Max angle")
    #axes[1, 0].plot(x_ang, mean_ang, marker="o", label="Mean angle")
    axes[1, 0].axvline(axis_shift_step, linestyle="--")
    axes[1, 0].set_title("Consecutive manifold alignment")
    axes[1, 0].set_xlabel("Next window start step")
    axes[1, 0].set_ylabel("Principal angle (deg)")
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, "Not enough windows", ha="center", va="center")
    axes[1, 0].set_title("Consecutive manifold alignment")
    axes[1, 0].set_axis_off()

# Optional combined z-scored view
def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    s = x.std()
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)

axes[1, 1].plot(x_win, zscore_safe(d95), marker="o", label="d95")
axes[1, 1].plot(x_win, zscore_safe(var_amb), marker="o", label="var ambient")
axes[1, 1].plot(x_win, zscore_safe(var_pca), marker="o", label="var pca")
if len(x_ang) > 0:
    axes[1, 1].plot(x_ang, zscore_safe(max_ang), marker="o", label="max angle")
axes[1, 1].axvline(axis_shift_step, linestyle="--")
axes[1, 1].set_title("Metrics together (z-scored)")
axes[1, 1].set_xlabel("Step")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


# --------------------------------------------------
# 2) Global PCA over all trials + trajectories + both decoder axes
# --------------------------------------------------
global_pca = fit_global_pca(
    snapshots=trainer.history.snapshots,
    phase=None,
    use_bci_only=use_bci_only,
    n_components=3,
)

X_pca = global_pca["X_pca"]              # [n_trials, T, 3]
A_pca = global_pca["A_pca"]              # [n_trials, 3]
snaps = global_pca["selected_snapshots"]
steps = np.array([s["step"] for s in snaps])

# Identify axis 1 and axis 2 from snapshots before/after shift
pre_idx = np.where(steps <= axis_shift_step)[0]
post_idx = np.where(steps > axis_shift_step)[0]

if len(pre_idx) == 0 or len(post_idx) == 0:
    raise ValueError("Could not find snapshots on both sides of axis_shift_step.")

axis1_idx = pre_idx[-1]
axis2_idx = post_idx[0]

axis1_pca = A_pca[axis1_idx]
axis2_pca = A_pca[axis2_idx]

# trials to plot
plot_idxs = list(range(0, len(snaps), every_k_trials))
if (len(snaps) - 1) not in plot_idxs:
    plot_idxs.append(len(snaps) - 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PC1-PC2
for i in plot_idxs:
    traj = X_pca[i]
    color = None
    if steps[i] <= axis_shift_step:
        color = "C0"
    else:
        color = "C1"

    axes[0].plot(traj[:, 0], traj[:, 1], alpha=0.35, linewidth=1, color=color)
    axes[0].scatter(traj[0, 0], traj[0, 1], s=15, color=color)
    axes[0].scatter(traj[-1, 0], traj[-1, 1], s=15, color=color)

axes[0].arrow(
    0, 0, axis1_pca[0], axis1_pca[1],
    head_width=0.05, length_includes_head=True, linewidth=2, color="black"
)
axes[0].arrow(
    0, 0, axis2_pca[0], axis2_pca[1],
    head_width=0.05, length_includes_head=True, linewidth=2, color="red"
)
axes[0].set_title("Global PCA: trajectories + decoder axes (PC1-PC2)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# PC1-PC3
for i in plot_idxs:
    traj = X_pca[i]
    color = None
    if steps[i] <= axis_shift_step:
        color = "C0"
    else:
        color = "C1"

    axes[1].plot(traj[:, 0], traj[:, 2], alpha=0.35, linewidth=1, color=color)
    axes[1].scatter(traj[0, 0], traj[0, 2], s=15, color=color)
    axes[1].scatter(traj[-1, 0], traj[-1, 2], s=15, color=color)

axes[1].arrow(
    0, 0, axis1_pca[0], axis1_pca[2],
    head_width=0.05, length_includes_head=True, linewidth=2, color="black"
)
axes[1].arrow(
    0, 0, axis2_pca[0], axis2_pca[2],
    head_width=0.05, length_includes_head=True, linewidth=2, color="red"
)
axes[1].set_title("Global PCA: trajectories + decoder axes (PC1-PC3)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC3")

plt.tight_layout()
plt.show()

print("Explained variance ratio:", global_pca["explained_variance_ratio"])
print("Axis1 step:", steps[axis1_idx], " Axis2 step:", steps[axis2_idx])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1x3 grid:
#   PC1 vs PC2
#   PC1 vs PC3
#   PC2 vs PC3
#
# Each trajectory:
# - color-coded by decoder-axis direction
# - shaded along time
# - transition points marked
# - decoder axis drawn from trajectory start
# ------------------------------------------------------------

X_pca = np.asarray(global_pca["X_pca"])          # [n_trials, time, n_pca]
A_pca = np.asarray(global_pca["A_pca"])          # [n_trials, n_pca]
epoch_ids = np.asarray(global_pca["epoch_ids"])  # [time]
evr = np.asarray(global_pca["explained_variance_ratio"])

# choose trials
trial_indices = np.arange(X_pca.shape[0])  # or e.g. np.arange(20)

X3 = X_pca[trial_indices, :, :3]   # [n_trials, T, 3]
A3 = A_pca[trial_indices, :3]      # [n_trials, 3]

def normalize_rows(x, eps=1e-12):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

def epoch_transition_indices(epoch_ids):
    return np.where(np.diff(epoch_ids) != 0)[0] + 1

# color each trajectory by decoder-axis direction angle in PC1-PC2 plane
A3_unit = normalize_rows(A3)
angles = np.arctan2(A3_unit[:, 1], A3_unit[:, 0])     # [-pi, pi]
angles01 = (angles + np.pi) / (2 * np.pi)             # [0, 1]
base_colors = plt.cm.hsv(angles01)

trans_idx = epoch_transition_indices(epoch_ids)

# arrow scale
all_pts = X3.reshape(-1, 3)
extent = np.percentile(np.linalg.norm(all_pts, axis=1), 95)
axis_scale = 0.25 * extent

pairs = [
    (0, 1, "PC1", "PC2", evr[0], evr[1]),
    (0, 2, "PC1", "PC3", evr[0], evr[2]),
    (1, 2, "PC2", "PC3", evr[1], evr[2]),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (i1, i2, lab1, lab2, var1, var2) in zip(axes, pairs):
    for i in range(len(trial_indices)):
        pts = X3[i]
        axis_vec = A3_unit[i]
        c = base_colors[i]
        T = pts.shape[0]

        # time-shaded trajectory
        for t in range(T - 1):
            frac = t / max(T - 2, 1)
            alpha = 0.15 + 0.85 * frac
            ax.plot(
                pts[t:t+2, i1],
                pts[t:t+2, i2],
                color=(c[0], c[1], c[2], alpha),
                linewidth=2,
            )

        # start and end
        ax.scatter(pts[0, i1], pts[0, i2], color=c, s=18, alpha=0.7)
        ax.scatter(pts[-1, i1], pts[-1, i2], color=c, s=28, alpha=1.0)

        # transition markers
        for ti in trans_idx:
            ax.scatter(
                pts[ti, i1], pts[ti, i2],
                color=c, s=55,
                edgecolor="k", linewidth=0.7, alpha=0.95
            )

        # decoder axis from trajectory start
        start = pts[0]
        vec = axis_scale * axis_vec
        ax.arrow(
            start[i1], start[i2],
            vec[i1], vec[i2],
            color=c,
            width=0.0,
            head_width=0.03 * extent,
            length_includes_head=True,
            alpha=0.95,
        )

    ax.set_xlabel(f"{lab1} ({100*var1:.1f}%)")
    ax.set_ylabel(f"{lab2} ({100*var2:.1f}%)")
    ax.set_title(f"{lab1} vs {lab2}")
    ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.show()